# Build the Sales example with Weaver

This notebook carries the example repository in its own source because the public Fabric Notebook definition API does not transport Notebook Resources. Attach the Weaver control Lakehouse and the published Weaver Environment, then run it. The build infers its workspace and control Lakehouse from that session; no workspace string is passed.

In [ ]:
import base64
import io
import json
from pathlib import Path
import tempfile
import zipfile

import weaver
from notebookutils import notebook

REPOSITORY_ARCHIVE = """UEsDBBQACAAIAIqFBF0AAAAAAAAAAKoDAAArAAAATGFrZWhvdXNlL1NhbGVzL0ZpbGVzL1NhbGVzX19PcmRlckV4cG9ydC5weW1SwWrbQBC971cM6qF2sVWTQgmCnOoaAk1akkMopZi1dmQtXe2K3ZEd0fbfO7OyG1Oqg0CaN2/emzdFUahNcAYj3K4reNQOU/k58vfH5z5EUmqNqY62Jxt8BQ/6CEGqgLkMjeUG0AkMOnvAiAZ2I1CLkIQK0pgIu1KpT9aj3mMF93bfkhvBxNBDE0P3P/SGaeEHjhUUb8o6HQqlbn0dsUNP2lVAcUCl7gNhquCXAthMOmJkEeB1xzqyzu1Xfu7u1mthAe0NYxAikmY9Bqw32FhvCd24gB3WekjIdP9KYi3YJwjeiTkbifXrMZVwoQqaELmoiel1km1pcEEbptPGpMzpxfzrlNeW1XRDIvBBesiyMgEFz3N3yHQIlngbD3iwifcvQDbMhEu4Wl29X66ul6t38IHnEZpSFRymyivtNbXO7sB2OaUv/HmqHFFzTOfCFL1SqnY6pSn+7fYi/9mEmMtQgFcVPIlBLRYdZ967MIp7aJDqlmW/BBqGWOM5UXhEOh9GrZ3jmSc+62Whie9Jd72TvgzCROwJWrksTRoocF+MwhDDsG8nppZTLDOT9XXorN/DDfz8rfIvTlZkmllC15wMyMPM+wkpa8nV8vRv22S3s/n8L1pClXtaAOEzidzccJ5X8ul0aXZBL8/sPONt7p2Xx8i4rTDM5LUA5H7DiJtioGZ5XbwM5EMYIk+heGaZL+Dbd/UHUEsHCGEHrb8VAgAAqgMAAFBLAwQUAAgACACKhQRdAAAAAAAAAACxAgAAJgAAAExha2Vob3VzZS9TYWxlcy9TYWxlcy5PcmRlclN1bW1hcnkuc3FshZLBTsMwDIbvfgofOLRT140hoSkSB7RdkJBAbA/QLPVKoImrJEXa25MMpmWARG6x+//+v7izCWzlrid8WAvcyJ58/eRacpvRGOkOAGvyyukhaLYCjy0MHGTvcXdANfrAhlyFe3boaGAXtO1qgEdtSXYk8CozBXh2OtniOx0Err7VqNs0ZyDbklWavADEaZ4mu59EACs2g3Tas0XF/WisPwVUPNpQ4TblRGnSDWCjXsnIZJ3NFeiDi4nzqpWGsnpmKXCnOx3N8MJbYEsqcvXF9bJalAAv9KF9fDC0HE40i/nidjpfTuc3uHIkA7U1TGYAnnpSAbluslhNlMRTocrKKde5kQYXkxL/OdJjkxGc9dKHwo+miIPvjxRNmT6+ICm/9DlrMtg7Nvl2kGPxjbX9sSJUmHZzSYZ3v1k7x+OQ/qcfnb/4PwFQSwcIK6kwa0YBAACxAgAAUEsDBBQACAAIAIqFBF0AAAAAAAAAAIMDAAAiAAAATGFrZWhvdXNlL1NhbGVzL1NhbGVzX19DdXN0b21lci5weW1SUWvbMBB+1684xB4cSLzQwRiGPSUrFEpb2r6V0qjSuRGxJXOSkwX243eSndTtphfju+/77u67k1KKR/XaIFytK3hQDYZy1YfoWyQh1hg02S5a7yq4dQjkD9AhgR4hsHP+4CB6iFuEkOgQjiFiWwpxbR2qN6zgy6XlxNdB/ZYM0q/fnacoxB3ZVtERdnis4FQXrBFi5dtOkQ3egfZN37owATjVohAPeoutqgRMmRWESNa9TaMJfo6LGx+Rxf4w4h6VgZp8m9vH3BTUvuEOgRTHiBPKDRAFwfekcRwQjCXUsTnOOZ74rBezkzYA4WtvG5N/M7lODiSxCAck5IG7yBbd494GdhdcboolFnCxvPi+WP5YLL/BilBFNKWQvCaRhbKVZbby5WXiJdg2f/7NjMQDqn1yaIDlnQshdKNCOJFOhhU5O0vtABiseRxlioBNPcbSG9xCAz//U3MAlx17WMzOFL6ewPDiHEgvIcvAu96VqUzp87kVcss/SHIOj9Tj7AOl1GFfnOp/SrEcb4W7oEJuJnexYaVJIJ3ERn7iGvLduu8aq9n1UDzJCV8+v4MnE2HsyeXB5tNJdF7cWkV1SVypeHr+UJ3bGc+RW/gLUEsHCAMHxx3NAQAAgwMAAFBLAwQUAAgACACKhQRdAAAAAAAAAABdBQAAHwAAAExha2Vob3VzZS9TYWxlcy9TYWxlc19fT3JkZXIucHl9U+9r2zAQ/a6/4vAGdcD1ug5GMeugpBsUxjrWfm8U+VKLylKQ5GSB/fF7shPH/cH8IbHu3r17unvOskzcy6Vhurmu6E4aDuWtr9kLcc1Beb2O2tmKbi2Td1tasyfl7Er7lmtSXYiuRcilklKIH9qyfOSK3n/XoPowIfz2Z+18FOKX1630O3riHWhThnQtxI1Vnlu2UZqKou9YiLlr19Lr4Cxamq61oaKr1nU2FvvCEGXsghB3quFWVoJGwgo5r+0jQvODyGfRAVjLCLHpF6GBG0dWkGjyjxfF+WyEDr1GBvHTRcbxLwAT7bRkJbvAFBsm7q9MOpCkrbY15oe7eFaAklRRb3TEEKQd5gemVocAdlp511J0Vj828STQCsOkRgayLqIDYyDSKjaG64J03yHodm125EzSGhtpwZY0DH1Lmg8FMq0TerzXGyYwojlkGq1AU7PhyKRtiCxrrPM3b3RAQeqLy4LylM7Pzj+fnl2cnn2iuWdMDsAMLhK95n7tZb/2h4fJ3gny0t/rzL5wy3KTdjTAeksKIZSRITwryvvULGkhCF5hnLLOA5vVPpaeYfAw6OUbDQdwuZaxyWdjCcwdAE+5MsB2T2UiLl3v/zxrcGCfFXQPa85KFTb5ocmsRBGrCHafj3zpyRYHOy6y4kVm4spXSSVDzBdHhy7SotLLLL1ME29XDkYeqqZeHsr32VeKpjafZo8zekdXR+MNnk3W8xy1R+DgI7Mr6WoZGEDaus7UvW1rB6fi0250mBAG13nAwKKPH1GBMMk9g7OwdcsSrs1cF4OueWLsrBzJjsou+22W24Y95y8uhuTJiDzJDsvLp7uauIJj5+3/6L58fc5XHHWIf1BLBwhBkSZBhAIAAF0FAABQSwMEFAAIAAgAioUEXQAAAAAAAAAA+wEAABwAAABMYWtlaG91c2UvU2FsZXMvbGliL2RhdGVzLnB5dZAxT8QwDIX3/AqrC3cSV4n1JJCQWNiRGK9u49KgJK5i99D9e5yowEQmx3l+73O6rntBJVgorlQEZMFCHsYb6BIEglK6E+DxkyaV3rlnWCOGDIn9FukeMisgvBNeqeyys02Bpymak1TBEvIHYPaQLKkEjEFI3P7Qw6uCJZk8JEtGMb/IaFVRmnEy/xiNhkAwWaJw9bcJT2vkG3k3kgRPTbKDWo2mSSsXreoaXijx1QK+Fsq1J8qrwEgNbtOFbe/edV3n3Fw4gTdYNaQfm3q/h9rxFBWdc55m+4esy0XUYA+cz011hNNTK84O7Jjlm6HNoUh1uQHPjbWNwjBwHgaYbUn77tw3gDpWSLeSgXNfbFGc6GCzjw/HPXgtdA28ycW6/yX/WsDpD7zaSPX5BlBLBwjPvU2SLQEAAPsBAABQSwMEFAAIAAgAioUEXQAAAAAAAAAAVAAAACEAAABMYWtlaG91c2UvU2FsZXMvc2NoZW1hcy9TYWxlcy55bWwVybERgCAMBdCeKf4AngPQSmPNBAjxyB0BL4H91eoVL+ZKknAGj5gamXOBLCs/k0f3OJbNIaS2YWj5ROoFsxJsiSRlMlyL28StQ/6Q3b1QSwcI7rK6S04AAABUAAAAUEsDBBQACAAIAIqFBF0AAAAAAAAAAJ0EAAAxAAAAV2FyZWhvdXNlL1JlcG9ydGluZy9SZXBvcnRpbmcuQ3VzdG9tZXJSZXZlbnVlLnNxbGVUwW4TMRC9+yvmgEQbpWkIAlXLra2QKiFAbW9VpDreSWJY22HsTYnEgY/gC/slPHs3yRZySmbevJn3ZpzzkbrXi4bp5rqiW94ESdavJldtTMGx3PKWfctKXXM0YjfJBp9xJUobFjI9ckxPNq1JU2xFwkonpu+8o2UQSmumyxtyoeZmotQn61mvuKJXx35fpGa5a53TslPqq9j8JRNUtB+FbK3UTc0+2TQMA6TUVXAbLTYGTyY0rfNxgPDa8ZhKC2Rbn8Z0H5JuSLv8S6k7s2anK0XDZhVttZi1lpN309NhKtMdk7NpyQ7YK1rYlQUvvWhTUc0GupqTNxfj2alSn0NijPkr4+DQ5iiabCymjRZthFkxjnJ0TNrXJW57F3qtGQ7fGQ7AdtAdV/Cf/Wil0x6LZeZaH1KuTzucgVkHayAvF7GHjMynY7QrH1/2zrU+ssDNGErqR8uY3+M4BGpC3RqOZFMZW1MTdA0up5NZY+WlPmXHmrqvWdoM9MQ/bcxXQRKeJsWbvQ97c54gAlzdLEvLWT/ApI0EoHKr+KHMdLSiLytKtVAsO6dvASIwC+6S6LK1GAZrarTwP3Ifu6UeAo+HbXQ2HUyynSEafHmOXltnFbI+BUA6WVvdtHl2YfqoF2LNazQNeQchMj3//oNiPCoWcJkABtMmu2UsTCjkg+O6oNAOOyyuOr0jYbSNjHV4w11HkDhIxMO1MW/dl9sD7RnNprP3Z9OLs+lbukJl4nqiRudKRW7Y4FQmD4NHMUcJPuMX4fwghonBWxiGh48hx5cSHD0c/gPmfWH/LzCnqP4CUEsHCMsNBHZgAgAAnQQAAFBLAwQUAAgACACKhQRdAAAAAAAAAABaAQAAHQAAAFdhcmVob3VzZS9SZXBvcnRpbmcvYWxpYXMueW1sVY+9asNAEIT7e4oBFW6C0jtVIKUhJDKkXuk20cX3Y25XCL991jpIlGYPvpud2enwjBNdeC6LMJTGyA9I5BnkfWWRO0HIEgzpzPig2sS961yH8x5hopyLojJ5vHBUwmeILE8IDcpm8Zt3EAxvJ3D21xKymqNtTZFqyF+bkmIgQRCsMykiq4B2cWX85kmRKZl0V6Nxc1uDGlGwvVztCk645LLe/VcjrVJpn4KxLNn3bktlOTrgna+lqsn71+q5DktKVG/Hv6zHgaxgm/807gdQSwcI8ulCl84AAABaAQAAUEsDBBQACAAIAIqFBF0AAAAAAAAAAFwAAAApAAAAV2FyZWhvdXNlL1JlcG9ydGluZy9zY2hlbWFzL1JlcG9ydGluZy55bWw1yrEOQDAUBdC9X3E/QHyAUbpYVWIubrRJefJe+X4Wy5lOWBOPiMF3GHmJ1nzuznnaqvmqWc4Oc1QmuY3Qf6DGpdAaGPXhhiroh08pBnmomHxo3QtQSwcIksPe3lQAAABcAAAAUEsBAh4DFAAIAAgAioUEXWEHrb8VAgAAqgMAACsAAAAAAAAAAQAAAKSBAAAAAExha2Vob3VzZS9TYWxlcy9GaWxlcy9TYWxlc19fT3JkZXJFeHBvcnQucHlQSwECHgMUAAgACACKhQRdK6kwa0YBAACxAgAAJgAAAAAAAAABAAAApIFuAgAATGFrZWhvdXNlL1NhbGVzL1NhbGVzLk9yZGVyU3VtbWFyeS5zcWxQSwECHgMUAAgACACKhQRdAwfHHc0BAACDAwAAIgAAAAAAAAABAAAApIEIBAAATGFrZWhvdXNlL1NhbGVzL1NhbGVzX19DdXN0b21lci5weVBLAQIeAxQACAAIAIqFBF1BkSZBhAIAAF0FAAAfAAAAAAAAAAEAAACkgSUGAABMYWtlaG91c2UvU2FsZXMvU2FsZXNfX09yZGVyLnB5UEsBAh4DFAAIAAgAioUEXc+9TZItAQAA+wEAABwAAAAAAAAAAQAAAKSB9ggAAExha2Vob3VzZS9TYWxlcy9saWIvZGF0ZXMucHlQSwECHgMUAAgACACKhQRd7rK6S04AAABUAAAAIQAAAAAAAAABAAAApIFtCgAATGFrZWhvdXNlL1NhbGVzL3NjaGVtYXMvU2FsZXMueW1sUEsBAh4DFAAIAAgAioUEXcsNBHZgAgAAnQQAADEAAAAAAAAAAQAAAKSBCgsAAFdhcmVob3VzZS9SZXBvcnRpbmcvUmVwb3J0aW5nLkN1c3RvbWVyUmV2ZW51ZS5zcWxQSwECHgMUAAgACACKhQRd8ulCl84AAABaAQAAHQAAAAAAAAABAAAApIHJDQAAV2FyZWhvdXNlL1JlcG9ydGluZy9hbGlhcy55bWxQSwECHgMUAAgACACKhQRdksPe3lQAAABcAAAAKQAAAAAAAAABAAAApIHiDgAAV2FyZWhvdXNlL1JlcG9ydGluZy9zY2hlbWFzL1JlcG9ydGluZy55bWxQSwUGAAAAAAkACQDkAgAAjQ8AAAAA"""

with tempfile.TemporaryDirectory(prefix="weaver-sales-example-") as temporary:
    repository = Path(temporary)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(REPOSITORY_ARCHIVE))) as archive:
        archive.extractall(repository)
    result = weaver.build(
        repository,
        bind=[
            "Lakehouse/Play_LH=Lakehouse/Sales",
            "Warehouse/Play_WH=Warehouse/Reporting",
        ],
    )

if not result.succeeded:
    raise RuntimeError(json.dumps(result.to_mapping(), indent=2))
notebook.exit(json.dumps(result.to_mapping()))
